# Invoice Extractor with MongoDB Storage

- **Valid invoices** → stored as JSON documents in `invoices.processed` collection
- **Invalid invoices** (failed extraction / low confidence) → stored as binary file in `invoices.invalid_files` using GridFS
- Invalid JSON responses are **never** written to MongoDB

In [ ]:
import ollama
import cv2
import json
import re
import os
import shutil
from datetime import datetime, timezone

from pydantic import BaseModel, ValidationError
from typing import List, Optional

# MongoDB
from pymongo import MongoClient
import gridfs

## MongoDB Setup

Two separate areas inside the `invoices` database:
| Area | Type | Purpose |
|---|---|---|
| `invoices.processed` | Collection | Valid invoice JSON documents |
| GridFS bucket `invalid_files` | GridFS | Raw image files of invalid/low-confidence invoices |

In [ ]:
# ── MongoDB connection ──────────────────────────────────────────────────────
MONGO_URI = "mongodb://localhost:27017"   # Change if using Atlas or remote host
DB_NAME   = "invoices"

mongo_client   = MongoClient(MONGO_URI)
db             = mongo_client[DB_NAME]

# Collection for valid, fully-extracted invoices
processed_col  = db["processed"]

# GridFS bucket for invalid invoice image files
invalid_fs     = gridfs.GridFS(db, collection="invalid_files")

print(f"Connected to MongoDB — database: '{DB_NAME}'")
print(f"  Valid invoices  → collection : {DB_NAME}.processed")
print(f"  Invalid files   → GridFS     : {DB_NAME}.invalid_files")

In [ ]:
# Local fallback folders (kept for compatibility / manual review)
REVIEW_FOLDER = "reviews"
os.makedirs(REVIEW_FOLDER, exist_ok=True)

PROCESSED_FOLDER = "processed"
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

## Image Preprocessing

Steps:
1. **2× upscale** — helps the vision LLM read small text (prices, dates, item codes)
2. **Grayscale conversion** — removes colour noise, reduces image size passed to model
3. **CLAHE** (Contrast Limited Adaptive Histogram Equalisation) — boosts legibility on faded or low-contrast invoices without blowing out bright areas

> `fastNlMeansDenoisingColored` was removed — it was designed for Tesseract-style OCR pipelines, is very slow (2–5 s/image), and can smear fine text that a vision LLM handles natively.

In [ ]:
def preprocess_image(image_path):
    image = cv2.imread(image_path)

    if image is None:
        print(f"Warning: Could not read image: {image_path}")
        return None

    # Step 1: Upscale — improves readability of small text for the vision LLM
    image = cv2.resize(image, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # Step 2: Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Step 3: CLAHE — enhance contrast locally without over-brightening
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    filename       = f"processed_{os.path.basename(image_path)}"
    processed_path = os.path.join(PROCESSED_FOLDER, filename)
    cv2.imwrite(processed_path, enhanced)

    return processed_path

## LLM Prompt & Extraction

In [ ]:
PROMPT = """
You are an information extraction system.

Extract the following fields from the invoice image:

{
  "title": string | null,
  "tax": number | null,
  "expenses": number,
  "date": string | null,
  "items": list[string]
}

CRITICAL DATE RULE:
If the date on the invoice is "12/05/24", "May 12, 2024", or "12-05-2024",
you MUST convert it to "2024-05-12" — always return date as YYYY-MM-DD.
If the year is missing, assume 2026.

STRICT RULES:
- Return ONLY valid JSON
- Do NOT guess values
- If unsure, return null
- expenses must be FINAL TOTAL
- tax must be numeric
- date format: YYYY-MM-DD
- items must be short descriptions
"""

In [ ]:
def call_qwen(image_path):
    response = ollama.chat(
        model='qwen3.5:4b',
        messages=[{
            'role': 'user',
            'content': PROMPT,
            'images': [image_path]
        }],
        format='json'
    )
    return response['message']['content']

## Validation

In [ ]:
class Invoice(BaseModel):
    title:    Optional[str]
    tax:      Optional[float]
    expenses: float
    date:     Optional[str]
    items:    List[str]

In [ ]:
def validate_json(response_text):
    """
    Returns an Invoice Pydantic model on success, or None on failure.
    Invalid JSON strings are NEVER passed further — they return None here.
    """
    try:
        data = json.loads(response_text)
        return Invoice.model_validate(data)
    except (json.JSONDecodeError, ValidationError) as e:
        print(f"  JSON/validation error: {e}")
        return None

In [ ]:
def business_validation(data: Invoice):
    if data.expenses <= 0:
        return False
    if data.tax is not None and data.tax < 0:
        return False
    if data.date:
        if not re.match(r"\d{4}-\d{2}-\d{2}", data.date):
            return False
    return True

In [ ]:
def extract_with_retry(image_path, max_retries=2):
    for attempt in range(max_retries):
        result    = call_qwen(image_path)
        validated = validate_json(result)          # invalid JSON → None, never stored
        if validated and business_validation(validated):
            print(f"  Extraction succeeded on attempt {attempt + 1}")
            return validated
        print(f"  Retrying... attempt {attempt + 1}")
    return None

In [ ]:
def verify_output(image_path, data: Invoice):
    checks = []

    if data.tax is not None and data.items:
        if data.tax > 1:
            implied_subtotal = data.expenses - data.tax
            checks.append(implied_subtotal > 0)

    checks.append(0 < len(data.items) <= 50)
    checks.append(0 < data.expenses < 1_000_000)

    if data.date:
        try:
            datetime.strptime(data.date, "%Y-%m-%d")
            checks.append(True)
        except ValueError:
            checks.append(False)

    return all(checks) if checks else False

In [ ]:
def compute_confidence(data, verified):
    score    = 0
    possible = 0

    possible += 0.4
    if data.expenses and data.expenses > 0:
        score += 0.4

    possible += 0.1
    if data.tax is not None:
        score += 0.1

    possible += 0.15
    if data.date and re.match(r"\d{4}-\d{2}-\d{2}", data.date):
        score += 0.15

    possible += 0.1
    if data.title and len(data.title.strip()) > 2:
        score += 0.1

    possible += 0.15
    if data.items and len(data.items) > 0:
        score += 0.15

    possible += 0.1
    if verified:
        score += 0.1

    return round(score / possible, 2) if possible else 0.0

## MongoDB Write Helpers

- `save_valid_invoice()` → inserts the structured JSON document into `invoices.processed`
- `save_invalid_file()` → stores the raw image binary in GridFS (`invoices.invalid_files`)

> **Note:** invalid JSON strings are dropped before reaching either function.

In [ ]:
def save_valid_invoice(image_path: str, data: Invoice, confidence: float) -> str:
    """
    Persist a successfully extracted invoice as a JSON document
    in the `invoices.processed` collection.

    Returns the inserted document's _id as a string.
    """
    document = {
        "source_file":  os.path.basename(image_path),
        "confidence":   confidence,
        "extracted_at": datetime.now(timezone.utc),
        "invoice":      data.model_dump()          # plain dict — no raw JSON strings
    }

    result = processed_col.insert_one(document)
    print(f"  [MongoDB] Valid invoice saved → _id: {result.inserted_id}")
    return str(result.inserted_id)

In [ ]:
def save_invalid_file(image_path: str, reason: str, confidence: float = 0.0) -> str:
    """
    Store the raw invoice image in GridFS under the `invalid_files` bucket.
    Metadata records why it failed.

    NOTE: No JSON payload is written here — only the binary image file.

    Returns the GridFS file _id as a string.
    """
    filename = os.path.basename(image_path)

    with open(image_path, "rb") as f:
        file_id = invalid_fs.put(
            f,
            filename    = filename,
            metadata    = {
                "reason":       reason,
                "confidence":   confidence,
                "uploaded_at":  datetime.now(timezone.utc),
                "source_path":  image_path
            }
        )

    print(f"  [MongoDB] Invalid file stored in GridFS → _id: {file_id}  reason: {reason}")
    return str(file_id)

## Local Review Fallback (unchanged)

In [ ]:
def send_to_review(image_path, result_json: str = None):
    """Copy the image (and optional valid JSON sidecar) to the local reviews folder."""
    filename    = os.path.basename(image_path)
    destination = os.path.join(REVIEW_FOLDER, filename)
    shutil.copy(image_path, destination)

    # Only write JSON sidecar when result_json is a *valid* serialised Invoice
    if result_json:
        base = os.path.splitext(filename)[0]
        with open(os.path.join(REVIEW_FOLDER, base + ".json"), "w") as f:
            f.write(result_json)

    print(f"  [Review] Copied to local reviews folder: {destination}")

## Main Pipeline

```
Image
  └─► preprocess
        └─► extract (with retry)       ← invalid JSON silently dropped here
              └─► verify + confidence
                    ├─ HIGH confidence + verified  →  MongoDB processed collection
                    └─ LOW confidence / unverified →  MongoDB GridFS (image only)
```

In [ ]:
def process_invoice(image_path: str) -> dict:
    print(f"\nProcessing: {image_path}")

    # ── Step 1: Preprocess ────────────────────────────────────────────────
    processed_img = preprocess_image(image_path)

    if processed_img is None:
        reason = "Could not preprocess image"
        send_to_review(image_path)
        save_invalid_file(image_path, reason=reason)   # image → GridFS, no JSON
        return {"status": "failed", "message": reason}

    # ── Step 2: Extract (invalid JSON never leaves extract_with_retry) ────
    result = extract_with_retry(processed_img)

    if not result:
        reason = "Extraction failed after all retries"
        send_to_review(image_path)
        save_invalid_file(image_path, reason=reason)   # image → GridFS, no JSON
        return {"status": "failed", "message": reason}

    # ── Step 3: Verify ───────────────────────────────────────────────────
    verified   = verify_output(processed_img, result)

    # ── Step 4: Confidence ───────────────────────────────────────────────
    confidence = compute_confidence(result, verified)

    # ── Step 5: Route to MongoDB ─────────────────────────────────────────
    if confidence < 0.85 or not verified:
        reason = f"Low confidence ({confidence}) or failed verification"
        send_to_review(image_path, result.model_dump_json())
        save_invalid_file(                              # image → GridFS, no JSON
            image_path,
            reason=reason,
            confidence=confidence
        )
        return {
            "status":     "low_confidence",
            "confidence": confidence,
            "verified":   verified
        }

    # ── Valid invoice → structured JSON document in MongoDB ───────────────
    mongo_id = save_valid_invoice(image_path, result, confidence)

    return {
        "status":     "success",
        "confidence": confidence,
        "mongo_id":   mongo_id,
        "data":       result.model_dump()
    }

## Query Helpers

Convenience functions to inspect what's in MongoDB.

In [ ]:
def list_valid_invoices(limit: int = 10):
    """Print the most recently processed valid invoices."""
    docs = processed_col.find().sort("extracted_at", -1).limit(limit)
    for doc in docs:
        inv = doc["invoice"]
        print(f"  _id={doc['_id']}  file={doc['source_file']}  "
              f"confidence={doc['confidence']}  "
              f"expenses={inv.get('expenses')}  date={inv.get('date')}")

In [ ]:
def list_invalid_files(limit: int = 10):
    """Print files stored in the invalid GridFS bucket."""
    for gf in invalid_fs.find().sort("uploadDate", -1).limit(limit):
        meta = gf.metadata or {}
        print(f"  _id={gf._id}  file={gf.filename}  "
              f"reason='{meta.get('reason')}'  "
              f"confidence={meta.get('confidence')}  "
              f"uploaded={gf.upload_date}")

In [ ]:
def retrieve_invalid_image(file_id_str: str, save_to: str = "."):
    """
    Download an invalid invoice image from GridFS back to disk.

    Usage:
        retrieve_invalid_image("64abc123...", save_to="/tmp")
    """
    from bson import ObjectId

    gf       = invalid_fs.get(ObjectId(file_id_str))
    out_path = os.path.join(save_to, gf.filename)

    with open(out_path, "wb") as f:
        f.write(gf.read())

    print(f"Retrieved: {out_path}")
    return out_path

## Run

In [ ]:
# --- Process a valid invoice ---
result = process_invoice(
    r"C:\Users\r1005\.cache\kagglehub\datasets\osamahosamabdellatif"
    r"\high-quality-invoice-images-for-ocr\versions\3\batch_1\batch_1\batch1_1\batch1-0001.jpg"
)
print(result)

In [ ]:
# --- Process an image that will likely fail ---
result = process_invoice(r"C:\Users\r1005\Documents\richard whitebg.png")
print(result)

In [ ]:
# --- Inspect what's in MongoDB ---
print("=== Valid Invoices (processed collection) ===")
list_valid_invoices()

print("\n=== Invalid Invoice Files (GridFS) ===")
list_invalid_files()